# Part 1 (Spoof Detection)

## Step 1: Setup and Installation

In [ ]:
# Install and upgrade required libraries
!pip install -q --upgrade transformers datasets evaluate accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 20.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


## Step 2: Load and Prepare the Dataset


In [ ]:
from datasets import load_dataset, concatenate_datasets, Image
import torch

# Load the dataset from its only available split: 'test'
full_dataset = load_dataset("nguyenkhoa/celeba-spoof-for-face-antispoofing-test", split='test')

# Create a smaller, balanced subset for faster training
subset_size = 4000
real_samples = full_dataset.filter(lambda x: x['labels'] == 0).shuffle(seed=42).select(range(subset_size // 2))
spoof_samples = full_dataset.filter(lambda x: x['labels'] == 1).shuffle(seed=42).select(range(subset_size // 2))

# Combine and shuffle them
combined_dataset = concatenate_datasets([real_samples, spoof_samples]).shuffle(seed=42)

# --- THE CRUCIAL CLEANING STEP ---
# Remove any rows where the image data might be None or corrupted
cleaned_dataset = combined_dataset.filter(lambda example: example["cropped_image"] is not None)
print(f"Original size: {len(combined_dataset)}. Cleaned size: {len(cleaned_dataset)}")


# Split this cleaned dataset into our training and testing sets
dataset = cleaned_dataset.train_test_split(test_size=0.2)

# Rename columns for consistency
dataset = dataset.rename_column("cropped_image", "image")
dataset = dataset.rename_column("labels", "spoof")

# Forcefully cast the column to the Image type to resolve any ambiguity
dataset = dataset.cast_column("image", Image())

print("\nDataset structure after cleaning and casting:")
print(dataset)

# Define labels
labels = ['real', 'spoof']
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

Filter:   0%|          | 0/4000 [00:00<?, ? examples/s]

Original size: 4000. Cleaned size: 3978

Dataset structure after cleaning and casting:
DatasetDict({
    train: Dataset({
        features: ['image', 'spoof', 'labelNames'],
        num_rows: 3182
    })
    test: Dataset({
        features: ['image', 'spoof', 'labelNames'],
        num_rows: 796
    })
})


### Diagnostic Cell

In [ ]:

# Run this cell immediately after Step 2 to check the data quality

print("--- Running Data Diagnostics on the 'train' split ---")
problem_found = False

# Check the first 500 samples
for i, example in enumerate(dataset["train"]):
    if i >= 500:
        break

    # The actual data is under the 'image' key
    image_data = example["image"]

    # Check if the data is None or not a valid PIL Image
    if image_data is None:
        print(f"!!! PROBLEM FOUND at index {i}: The image data is None.")
        problem_found = True
        break
    if not hasattr(image_data, 'convert'): # All PIL Images have a .convert method
         print(f"!!! PROBLEM FOUND at index {i}: The object is not a valid image. It's a {type(image_data)}")
         problem_found = True
         break

if not problem_found:
    print("--- Diagnostics Complete: No problems found in the first 500 samples. ---")
else:
    print("--- Diagnostics Complete: A data corruption issue has been identified. ---")

--- Running Data Diagnostics on the 'train' split ---
--- Diagnostics Complete: No problems found in the first 500 samples. ---


## Step 3: Fine-Tune the ViT Model


In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer, DefaultDataCollator
import evaluate
import numpy as np

# 1. Load ViT Image Processor
vit_checkpoint = "google/vit-base-patch16-224-in21k"
vit_processor = ViTImageProcessor.from_pretrained(vit_checkpoint)

# 2. Create a new transformation function for .map()
def process_vit(examples):
    # The processor handles resizing, normalization, and tensor conversion
    inputs = vit_processor(images=examples['image'], return_tensors='pt')
    # The trainer expects the label column to be named 'labels'
    inputs['labels'] = examples['spoof']
    return inputs

# 3. Apply the transformation to the entire dataset
# Apply the transformation and remove the original columns
prepared_ds_vit = dataset.map(process_vit, batched=True, remove_columns=["image", "spoof"])

# 4. Define Evaluation Metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    precision = precision_metric.compute(predictions=predictions, references=labels)["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels)["recall"]
    f1 = f1_metric.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# 5. Use the standard DefaultDataCollator
data_collator = DefaultDataCollator()

# 6. Load the ViT Model
vit_model = ViTForImageClassification.from_pretrained(
    vit_checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# 7. Set Training Arguments
training_args = TrainingArguments(
    output_dir="./vit-spoof-detector",
    report_to="none",
    per_device_train_batch_size=16,
    eval_strategy="steps",
    num_train_epochs=3,
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
)

# 8. Create and Run the Trainer
vit_trainer = Trainer(
    model=vit_model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=prepared_ds_vit["train"],
    eval_dataset=prepared_ds_vit["test"],
)

print("--- Starting ViT Model Training ---")
vit_trainer.train()

# 9. Evaluate the fine-tuned ViT model
print("\n--- Evaluating ViT Model ---")
vit_results = vit_trainer.evaluate()
print(vit_results)

Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

Map:   0%|          | 0/796 [00:00<?, ? examples/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Starting ViT Model Training ---


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,0.144400,0.444377,0.883166,0.993421,0.768448,0.866571
200,0.072600,0.031807,0.986181,0.997396,0.974555,0.985843
300,0.004300,0.060677,0.986181,0.997396,0.974555,0.985843
400,0.031500,0.115753,0.974874,0.953771,0.997455,0.975124
500,0.002300,0.025480,0.993719,0.989899,0.997455,0.993663



--- Evaluating ViT Model ---


{'eval_loss': 0.02548018842935562, 'eval_accuracy': 0.9937185929648241, 'eval_precision': 0.98989898989899, 'eval_recall': 0.9974554707379135, 'eval_f1': 0.9936628643852978, 'eval_runtime': 87.0666, 'eval_samples_per_second': 9.142, 'eval_steps_per_second': 1.149, 'epoch': 3.0}


## Step 4: Fine-Tune the SWIN Model


In [ ]:
from transformers import AutoImageProcessor, SwinForImageClassification, DefaultDataCollator

# 1. Load SWIN Image Processor
swin_checkpoint = "microsoft/swin-base-patch4-window7-224-in22k"
swin_processor = AutoImageProcessor.from_pretrained(swin_checkpoint)

# 2. Create a transformation function for .map()
def process_swin(examples):
    inputs = swin_processor(images=examples['image'], return_tensors='pt')
    inputs['labels'] = examples['spoof']
    return inputs

# 3. Apply the transformation
# Apply the transformation and remove the original columns
prepared_ds_swin = dataset.map(process_swin, batched=True, remove_columns=["image", "spoof"])

# 4. Use the standard DefaultDataCollator
data_collator = DefaultDataCollator()

# 5. Load the SWIN Model
swin_model = SwinForImageClassification.from_pretrained(
    swin_checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# 6. Set Training Arguments
training_args_swin = TrainingArguments(
    output_dir="./swin-spoof-detector",
    report_to="none",
    per_device_train_batch_size=16,
    eval_strategy="steps",
    num_train_epochs=3,
    save_steps=100,
    eval_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
)

# 7. Create and Run the Trainer
swin_trainer = Trainer(
    model=swin_model,
    args=training_args_swin,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=prepared_ds_swin["train"],
    eval_dataset=prepared_ds_swin["test"],
)

print("--- Starting SWIN Model Training ---")
swin_trainer.train()

# 8. Evaluate the fine-tuned SWIN model
print("\n--- Evaluating SWIN Model ---")
swin_results = swin_trainer.evaluate()
print(swin_results)

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Map:   0%|          | 0/3182 [00:00<?, ? examples/s]

Map:   0%|          | 0/796 [00:00<?, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

...

--- Starting SWIN Model Training ---


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,0.415200,0.146930,0.964824,0.933492,1.000000,0.965602
200,0.083400,0.133709,0.978643,1.000000,0.956743,0.977893
300,0.000000,0.014178,0.997487,0.997455,0.997455,0.997455
400,0.001300,0.044693,0.994975,0.997442,0.992366,0.994898
500,0.000000,0.015488,0.997487,0.997455,0.997455,0.997455



--- Evaluating SWIN Model ---


{'eval_loss': 0.014178466983139515, 'eval_accuracy': 0.9974874371859297, 'eval_precision': 0.9974554707379135, 'eval_recall': 0.9974554707379135, 'eval_f1': 0.9974554707379135, 'eval_runtime': 87.4334, 'eval_samples_per_second': 9.104, 'eval_steps_per_second': 1.144, 'epoch': 3.0}


## Step 5: Compare Model Performance


In [ ]:
import pandas as pd

# Create a dictionary to hold the results
comparison_data = {
    "ViT": {
        "Accuracy": vit_results["eval_accuracy"],
        "Precision": vit_results["eval_precision"],
        "Recall": vit_results["eval_recall"],
        "F1-Score": vit_results["eval_f1"],
    },
    "SWIN": {
        "Accuracy": swin_results["eval_accuracy"],
        "Precision": swin_results["eval_precision"],
        "Recall": swin_results["eval_recall"],
        "F1-Score": swin_results["eval_f1"],
    }
}

# Create and display a DataFrame for comparison
df_comparison = pd.DataFrame(comparison_data)

# Format the results to 4 decimal places for better readability
print("--- Model Performance Comparison ---")
print(df_comparison.round(4))

--- Model Performance Comparison ---
              ViT    SWIN
Accuracy   0.9937  0.9975
Precision  0.9899  0.9975
Recall     0.9975  0.9975
F1-Score   0.9937  0.9975


## Step 6: Test on Your Own Photos

In [ ]:
from google.colab import files
from PIL import Image
import io

# Create a prediction pipeline
def predict_image(model, processor, image_path):
    # Load and process the image
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(model.device)

    # Make a prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_idx = logits.argmax(-1).item()

    return model.config.id2label[predicted_class_idx]

# --- Test with your REAL photo ---
print("Please upload your REAL photo:")
uploaded_real = files.upload()

if uploaded_real:
    real_photo_path = list(uploaded_real.keys())[0]
    print(f"\nUploaded '{real_photo_path}'. Analyzing...")
    display(Image.open(real_photo_path).resize((224, 224)))

    # Predict with ViT
    vit_prediction_real = predict_image(vit_model, vit_processor, real_photo_path)
    print(f"👁️ ViT Model Prediction: {vit_prediction_real.upper()}")

    # Predict with SWIN
    swin_prediction_real = predict_image(swin_model, swin_processor, real_photo_path)
    print(f"👁️ SWIN Model Prediction: {swin_prediction_real.upper()}")
else:
    print("No file uploaded.")

print("\n" + "="*50 + "\n")

# --- Test with your SPOOFED photo ---
print("Please upload your SPOOFED photo (e.g., a photo of your face on a phone screen):")
uploaded_spoof = files.upload()

if uploaded_spoof:
    spoof_photo_path = list(uploaded_spoof.keys())[0]
    print(f"\nUploaded '{spoof_photo_path}'. Analyzing...")
    display(Image.open(spoof_photo_path).resize((224, 224)))

    # Predict with ViT
    vit_prediction_spoof = predict_image(vit_model, vit_processor, spoof_photo_path)
    print(f"🤖 ViT Model Prediction: {vit_prediction_spoof.upper()}")

    # Predict with SWIN
    swin_prediction_spoof = predict_image(swin_model, swin_processor, spoof_photo_path)
    print(f"🤖 SWIN Model Prediction: {swin_prediction_spoof.upper()}")
else:
    print("No file uploaded.")